# n8n run evaluation

Load the **most recent** saved n8n output under `outputs/n8n/`, then compare each author's **primary** answer to the **final** answer:

- **length** (characters and words)
- **ROUGE-1 / ROUGE-2 / ROUGE-L** (primary as reference, final as candidate)
- **TF-IDF cosine** 

For a merge run, final is the merged answer. For a judge run, final is the **selected refined version**.


In [1]:
from __future__ import annotations

import json
import re
from datetime import datetime
from pathlib import Path

import pandas as pd
from IPython.display import display, Markdown
from rouge_score import rouge_scorer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

OUTPUT_DIR = Path("outputs/n8n")
# Set to a specific file to override "most recent" (e.g. OUTPUT_DIR / "1f233aaa" / "judge_run.json")
RUN_PATH: Path | None = None

TFIDF_MAX_FEATURES = 4096
# Blind labels: "Version 5". Older runs used "LLM-5 (model-slug)" or "expert 5".
SELECTED_VERSION_RE = re.compile(
    r"(?:LLM\s*-?\s*|expert\s+|version\s+)(\d+)",
    re.I,
)


def _parse_saved_at(raw: str) -> datetime | None:
    if not raw:
        return None
    try:
        return datetime.fromisoformat(raw.replace("Z", "+00:00"))
    except ValueError:
        return None


def latest_run_path() -> Path:
    files = list(OUTPUT_DIR.glob("*/judge_run.json")) + list(
        OUTPUT_DIR.glob("*/roundtable_run.json")
    )
    if not files:
        raise FileNotFoundError(f"No n8n run JSON under {OUTPUT_DIR.resolve()}")

    def sort_key(p: Path):
        try:
            data = json.loads(p.read_text(encoding="utf-8"))
            ts = _parse_saved_at(data.get("saved_at_utc") or "")
        except (OSError, json.JSONDecodeError):
            ts = None
        return (ts is not None, ts or datetime.min, p.stat().st_mtime)

    return max(files, key=sort_key)


path = Path(RUN_PATH) if RUN_PATH else latest_run_path()
run = json.loads(path.read_text(encoding="utf-8"))
models = list(run.get("models") or [])
phase = run.get("phase") or "unknown"
display(
    Markdown(
        f"Loaded `{path}`  \n"
        f"Run `{run.get('run_id')}` · phase **{phase}** · "
        f"saved `{run.get('saved_at_utc')}` · {len(models)} authors"
    )
)

Loaded `outputs/n8n/1f233aaa/judge_run.json`  
Run `1f233aaa` · phase **judged** · saved `2026-08-18T17:25:50.625084+00:00` · 5 authors

In [2]:
def word_count(text: str) -> int:
    return len((text or "").split())


def tfidf_cosine(a: str, b: str) -> float:
    if not (a or "").strip() or not (b or "").strip():
        return 0.0
    vec = TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES,
        stop_words="english",
        smooth_idf=True,
        sublinear_tf=False,
        norm="l2",
    )
    try:
        m = vec.fit_transform([a, b])
        return float(cosine_similarity(m[0:1], m[1:2])[0, 0])
    except ValueError:
        return 0.0


def selected_refined(author_i: int, judgment: str, refined: list) -> str | None:
    """Map a judge rationale back to that author's chosen refined version."""
    if not judgment or not refined:
        return None
    lines = [ln for ln in judgment.splitlines() if ln.strip()]
    candidates = [ln for ln in lines if "select" in ln.lower()] + lines[:1] + [judgment]
    for text in candidates:
        match = SELECTED_VERSION_RE.search(text)
        if not match:
            continue
        critic_i = int(match.group(1)) - 1
        if critic_i == author_i or not (0 <= critic_i < len(refined)):
            continue
        picked = refined[critic_i]
        if isinstance(picked, str) and picked.strip():
            return picked
    return None


def final_for_author(i: int) -> tuple[str, str]:
    merged = (run.get("merged_responses") or [None] * len(models))
    judgments = (run.get("judgments") or [None] * len(models))
    refined_grid = (run.get("refined_responses") or [])
    merged_i = merged[i] if i < len(merged) else None
    if isinstance(merged_i, str) and merged_i.strip():
        return merged_i, "merged"
    judgment = judgments[i] if i < len(judgments) else None
    row = refined_grid[i] if i < len(refined_grid) else []
    picked = selected_refined(i, judgment or "", row or [])
    if picked:
        return picked, "judge-selected refine"
    if isinstance(judgment, str) and judgment.strip():
        return judgment, "judgment text"
    return "", "missing"


scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
primary = list(run.get("primary_responses") or [])
rows = []
for i, model in enumerate(models):
    p = (primary[i] if i < len(primary) else None) or ""
    final, source = final_for_author(i)
    rouge = scorer.score(p, final) if p.strip() and final.strip() else None
    rows.append(
        {
            "author": i + 1,
            "model": model,
            "final_source": source,
            "primary_chars": len(p),
            "final_chars": len(final),
            "primary_words": word_count(p),
            "final_words": word_count(final),
            "word_ratio_final_over_primary": (
                word_count(final) / word_count(p) if word_count(p) else None
            ),
            "rouge1_f": rouge["rouge1"].fmeasure if rouge else None,
            "rouge2_f": rouge["rouge2"].fmeasure if rouge else None,
            "rougeL_f": rouge["rougeL"].fmeasure if rouge else None,
            "tfidf_cosine": tfidf_cosine(p, final) if p.strip() and final.strip() else None,
        }
    )

df = pd.DataFrame(rows)
fmt = {
    "word_ratio_final_over_primary": "{:.2f}",
    "rouge1_f": "{:.3f}",
    "rouge2_f": "{:.3f}",
    "rougeL_f": "{:.3f}",
    "tfidf_cosine": "{:.3f}",
}
display(df.style.format(fmt, na_rep="—").hide(axis="index"))
numeric = ["primary_words", "final_words", "rouge1_f", "rouge2_f", "rougeL_f", "tfidf_cosine"]
means = df[numeric].mean(numeric_only=True)
display(
    Markdown(
        "**Means** · "
        + " · ".join(
            f"{k}: {v:.3f}" if not k.endswith("_words") else f"{k}: {v:.0f}"
            for k, v in means.items()
            if pd.notna(v)
        )
    )
)

author,model,final_source,primary_chars,final_chars,primary_words,final_words,word_ratio_final_over_primary,rouge1_f,rouge2_f,rougeL_f,tfidf_cosine
1,openai/gpt-5.6-sol,judge-selected refine,19201,24683,2769,3595,1.30,0.729,0.317,0.300,0.762
2,moonshotai/kimi-k3,judge-selected refine,23438,32931,3111,4462,1.43,0.581,0.155,0.142,0.550
3,qwen/qwen3.7-max,judge-selected refine,8481,8061,1236,1184,0.96,0.727,0.451,0.488,0.703
4,google/gemini-3.1-pro-preview,judge-selected refine,6991,8547,1046,1207,1.15,0.593,0.242,0.273,0.561
5,anthropic/claude-opus-4.8,judge-selected refine,7483,15002,996,2010,2.02,0.552,0.258,0.312,0.541


**Means** · primary_words: 1832 · final_words: 2492 · rouge1_f: 0.636 · rouge2_f: 0.285 · rougeL_f: 0.303 · tfidf_cosine: 0.623